# Download ERA5 offshore waves at the goals

ERA5 ocean waves (ECMWF; hourly, 0.5°, 1940 onwards) from the Earth Data Hub
zarr, read only at the goal nodes and written in the HyWaves names on
`(time, goal)`, like any other offshore source:

- bulk `hs`, `tp`, `dir`, `spr`;
- partitions `phs{i}`, `ptp{i}`, `pdir{i}`, `pspr{i}`: 0 = wind sea, 1–3 = the
  three swell partitions.

ERA5 has no partition peak periods: `ptp{i}` are the partition **mean**
periods (Tm-1,0), and the three swells share the total-swell spread. Fine for
the satellite calibration (Hs and directions); keep it in mind before using
ERA5 to force the dynamic runs or the reconstruction.

**Key:** set the `EDH_API_KEY` environment variable (e.g. in `~/.bashrc`), or
type it when asked below. Never write it in the notebook.

**Prerequisites:** `04_Select_Goals_by_Hindcasts.ipynb` (the goals JSON, with
an `era5` node per goal) and an `ERA5` entry in `offshore_waves`.
**Next:** `07b_Satellite_Calibration.ipynb` with `offshore_name = "ERA5"`.

In [1]:
import getpass
import os

import xarray as xr
from bluemath_tk.waves.hywaves.offshore import (
    EDH_KEY_VARIABLE,
    era5_at_goals,
    open_era5_waves,
)
from utils.config import get_offshore_wave, load_goals, resolve

## Configuration

In [3]:
offshore_name = "ERA5"  # offshore_waves entry this zarr feeds
period = slice("1979", "2023")  # time window (ERA5 starts in 1940)

out_path = resolve(get_offshore_wave(offshore_name)["data_path"])
if not os.environ.get(EDH_KEY_VARIABLE):
    os.environ[EDH_KEY_VARIABLE] = getpass.getpass("Earth Data Hub key: ")
print(f"output : {out_path}")
print(f"period : {period.start} – {period.stop}")

output : /lustre/geocean/WORK/users/javiert/projects/HyGloW/Netherlands/inputs/ERA5/processed/Netherlands_ERA5_goal.zarr
period : 1979 – 2023


## Goal nodes

The ERA5 node of each goal from the goals JSON (`hindcast.era5`, chosen in 04);
the nearest ERA5 grid point is read.

In [ ]:
goals = load_goals()
nodes = {gid: goal["hindcast"][offshore_name.casefold()] for gid, goal in goals.items()}
goal_ids = [int(gid) for gid in nodes]
lon = [node["longitude"] for node in nodes.values()]
lat = [node["latitude"] for node in nodes.values()]
list(zip(goal_ids, lon, lat))

## Download

All variables and partitions at the goal nodes (about a minute per decade).

In [ ]:
era5 = era5_at_goals(open_era5_waves(), lon, lat, goal_ids, time=period)
era5

In [ ]:
out_path.parent.mkdir(parents=True, exist_ok=True)
era5.chunk({"time": -1, "goal": 1}).to_zarr(out_path, mode="w")
print(f"written {out_path}")

## Check

In [4]:
check = xr.open_zarr(out_path)
summary = check[["hs", "phs0", "phs1", "phs2", "phs3"]].mean("time").to_dataframe()
summary["missing_hs"] = check["hs"].isnull().mean("time").to_series()
summary.round(3)

,hs,phs0,phs1,phs2,phs3,latitude,longitude,missing_hs
goal,,,,,,,,
1,1.087,0.842,0.496,0.137,0.033,51.5,2.5,0.0
2,1.225,0.935,0.564,0.156,0.034,52.0,3.0,0.0
3,1.295,0.971,0.646,0.177,0.037,52.5,3.5,0.0
4,1.353,1.000,0.664,0.175,0.037,53.0,4.0,0.0
5,1.403,1.024,0.807,0.223,0.048,53.5,4.5,0.0
6,1.567,1.118,0.852,0.236,0.051,54.0,5.0,0.0
7,1.540,1.108,0.828,0.215,0.046,54.0,5.5,0.0
8,1.495,1.080,0.799,0.191,0.042,54.0,6.0,0.0
9,1.446,1.042,0.777,0.175,0.038,54.0,6.5,0.0
